In [7]:
import asap3
import numpy as np
from ase.io import read
cutoff = 2.8
atom_list = read('/home/energy/mahpe/Published_code/cPaiNN/Relax_examples/one_hot_example.xyz',index=':')
atom_test = atom_list[1]
nl = asap3.FullNeighborList(cutoff, atom_test)
pair_i_idx = []
pair_j_idx = []
n_diff = []
for i in range(len(atom_test)):
    indices, diff, _ = nl.get_neighbors(i)
    pair_i_idx += [i] * len(indices)               # local index of pair i
    pair_j_idx.append(indices)   # local index of pair j
    n_diff.append(diff)

pair_j_idx = np.concatenate(pair_j_idx)
pairs_asap = np.stack((pair_i_idx, pair_j_idx), axis=1)
n_diff_asap = np.concatenate(n_diff)
print('Asap:',pairs_asap.shape, n_diff_asap.shape)
from scipy.spatial import distance_matrix
pos = atom_test.get_positions()
dist_mat = distance_matrix(pos, pos)
mask = dist_mat < cutoff
np.fill_diagonal(mask, False)        
pairs_ase = np.argwhere(mask)
n_diff_ase = pos[pairs_ase[:, 1]] - pos[pairs_ase[:, 0]]
print('Ase:',pairs_ase.shape, n_diff_ase.shape)

from matscipy.neighbours import neighbour_list
positions = atom_test.get_positions()
pbc = atom_test.get_pbc()
cell = atom_test.get_cell()
sender, receiver, unit_shifts = neighbour_list(
        quantities="ijS",
        pbc=pbc,
        cell=cell,
        positions=positions,
        cutoff=cutoff,
        # self_interaction=True,  # we want edges from atom to itself in different periodic images
        # use_scaled_positions=False,  # positions are not scaled positions
    )
edge_index = np.stack((sender, receiver)).T  # [2, n_edges]
shifts = np.dot(unit_shifts, cell)  # [n_edges, 3]
print('Matscipy:',edge_index.shape, shifts.shape)

# comapre asap and matscipy
print('Asap vs Matscipy')

# make the påaris in same order
pairs_asap = np.sort(pairs_asap, axis=0)
edge_index = np.sort(edge_index, axis=0)
np.array_equal(pairs_asap, edge_index)

Asap: (648, 2) (648, 3)
Ase: (454, 2) (454, 3)
Matscipy: (648, 2) (648, 3)
Asap vs Matscipy


True

In [4]:
atom_test.pbc

array([ True,  True,  True])